# 00 Filter the raw tweet file down to airline tweets

Reads the original Kaggle file `twcs.csv` (about 500 MB, 2.8 million rows) **in chunks**, so it never loads the whole file into memory, and writes `airline_tweets.csv`.

A row is kept if
- one of the nine airline support accounts wrote it, or
- it is a customer tweet that mentions one of those accounts (`@Delta`, `@AmericanAir`, and so on).

**Before running**
1. Put the original `twcs.csv` in a folder called `data` next to this notebook. Do not open or save it in Excel, which cuts it off at 1,048,575 rows.
2. Open the folder in VS Code and run the cells from top to bottom. Pick a Python kernel that has `pandas` installed.

In [ ]:
from pathlib import Path
import pandas as pd

RAW = Path("data/twcs.csv")                # the original Kaggle file
OUT = Path("data/airline_tweets.csv")      # the filtered file this notebook creates
CHUNKSIZE = 200_000                        # rows read at a time; lower it if memory is tight

print("raw file exists:", RAW.exists(), "|", RAW.resolve())

In [ ]:
AIRLINES = ["AirAsiaSupport", "AlaskaAir", "AmericanAir", "British_Airways", "Delta",
            "JetBlue", "SouthwestAir", "VirginAmerica", "VirginAtlantic"]
MENTION = "|".join("@" + a for a in AIRLINES)       # @AirAsiaSupport|@AlaskaAir|...
AIRLINES_LOWER = {a.lower() for a in AIRLINES}

EXPECTED_COLUMNS = ["tweet_id", "author_id", "inbound", "created_at", "text",
                    "response_tweet_id", "in_response_to_tweet_id"]
EXCEL_ROW_LIMIT = 1_048_575


def filter_airline_tweets(src, dst, chunksize=CHUNKSIZE):
    first, total_in, total_out = True, 0, 0
    for chunk in pd.read_csv(src, chunksize=chunksize, low_memory=False, dtype=str):
        if first:
            missing = [c for c in EXPECTED_COLUMNS if c not in chunk.columns]
            if missing:
                raise ValueError(f"The raw file is missing columns {missing}. Use the original "
                                 f"twcs.csv from Kaggle, not a copy saved or trimmed in Excel.")
        total_in += len(chunk)
        by_airline = chunk["author_id"].str.lower().isin(AIRLINES_LOWER)
        mentions = chunk["text"].fillna("").str.contains(MENTION, case=False, regex=True)
        keep = chunk[by_airline | mentions]
        keep.to_csv(dst, mode="w" if first else "a", header=first, index=False)
        first = False
        total_out += len(keep)
        print(f"read {total_in:,} rows, kept {total_out:,}", flush=True)
    if total_in == EXCEL_ROW_LIMIT:
        print("WARNING: exactly 1,048,575 rows, which is Excel's limit. The file was probably saved "
              "from Excel and cut off. Download twcs.csv again from Kaggle.")
    print(f"Done. {total_out:,} of {total_in:,} rows kept -> {dst}")
    return total_in, total_out

## Run the filter
Takes a few minutes on the full file. You will see one progress line per chunk.

In [ ]:
if not RAW.exists():
    raise FileNotFoundError(f"Cannot find {RAW.resolve()}. Put the original twcs.csv in the data folder.")
total_in, total_out = filter_airline_tweets(RAW, OUT)

## Check the result

In [ ]:
df = pd.read_csv(OUT, dtype=str)
print("rows:", f"{len(df):,}", "| columns:", list(df.columns))
print("duplicate tweet_id:", df["tweet_id"].duplicated().sum())

dates = pd.to_datetime(df["created_at"], format="%a %b %d %H:%M:%S %z %Y", utc=True, errors="coerce")
print("dates:", dates.min(), "to", dates.max(), "| unparsed:", dates.isna().sum())

print("\ntweets written by each airline account:")
print(df.loc[df["author_id"].str.lower().isin(AIRLINES_LOWER), "author_id"].value_counts().to_string())
print("\ncustomer tweets (inbound = True):", f"{(df['inbound'].str.lower() == 'true').sum():,}")

## What to expect on the full file
- The progress lines should reach about **2.8 million rows read** (the original file has 2,811,774).
- About **370,863 rows kept**, of which about 198,017 are customer tweets and 172,846 are airline replies.
- All seven columns are present, including `tweet_id` and `created_at`.
- If it reads exactly 1,048,575 rows, the file was cut off by Excel. Download it again.

Next step: `01_tweet_cleaning.ipynb` cleans `data/airline_tweets.csv` and builds the complaint-reply pairs and conversation transcripts.